# LABORATORIO N.° 03 — La métrica que importa

**Curso:** Analítica Empresarial Integrada  
**Semana 3:** KPI accionables, North Star Metric y árbol de métricas  
**Docente:** Pilar Rocío Sayán Mejía  
**Periodo:** 2026-II

**Fuente real:** UCI Machine Learning Repository — Online Retail (ID 352).

> Objetivo del notebook: conectar **objetivo → North Star → drivers → guardrails → KPI → tablero → interpretación → decisión** usando transacciones reales.
---

**Equipo N.°:** 2 &nbsp;&nbsp;&nbsp; **Sección:** C28 &nbsp;&nbsp;&nbsp; **Fecha:** 4/09/2026

**Apellidos y nombres del estudiante:** _(Pedro Sebastian Alfieri Arteaga Guerra)_

**Integrantes del equipo:** _()_

---


## Agenda de laboratorio — 7:00 p. m. a 10:10 p. m.

**Duración total:** 190 minutos · **Receso:** 15 minutos · **Trabajo efectivo:** 175 minutos.

| Horario | Tiempo | Desarrollo |
|---|---:|---|
| 7:00–7:10 | 10 min | Apertura del caso y presentación del problema de medición. |
| 7:10–7:35 | 25 min | Actividad 1. Revisión de conceptos: del dato a la decisión. |
| 7:35–8:00 | 25 min | Actividad 2. Descarga desde UCI, auditoría y limpieza documentada. |
| 8:00–8:30 | 30 min | Actividad 2. Periodo comparable, recurrencia y North Star. Reto 1. |
| 8:30–8:45 | 15 min | **RECESO** |
| 8:45–9:15 | 30 min | Actividad 2. Árbol de métricas, guardrails, Polars y DuckDB. |
| 9:15–9:35 | 20 min | Actividad 2. Tablero de decisión en Plotly. Reto 2. |
| 9:35–10:00 | 25 min | **Reto de aplicación y retroalimentación.** Ejercicios 1 a 5. |
| 10:00–10:10 | 10 min | Diccionario de KPI, informe ejecutivo y ticket de salida. |

> **Regla de trabajo:** no avance de bloque sin registrar la interpretación solicitada. El objetivo no es ejecutar celdas, sino convertir datos en evidencia para una decisión.


## Actividad 1 — Revisión de conceptos: del dato a la decisión (25 minutos)

**Propósito.** Establecer con precisión el vocabulario de medición antes de programar. La confusión entre dato, métrica, indicador y KPI constituye la causa más frecuente de tableros que no sustentan ninguna decisión.

**Instrucciones.** Complete la tabla con definiciones elaboradas con sus propias palabras. No se admite la reproducción literal de fuentes externas ni de sistemas generativos. La columna de la derecha contiene una pregunta de apoyo: si su definición permite responderla, la definición es suficiente; si no lo permite, corríjala antes de continuar.

**Evidencia esperada.** Tabla completa con las definiciones registradas.

| Concepto | Definición elaborada por el estudiante | Pregunta de apoyo |
|---|---|---|
| Dato | Es un valor o registro individual sin analizar, como el monto de una cuenta contable de una empresa en la base de la SMV. | ¿En qué se diferencia un dato de una métrica? Proponga un ejemplo de la base utilizada. |
| Métrica | Es un valor calculado a partir de datos, como el total de activos o la utilidad de una empresa. Solo es útil si ayuda a tomar una decisión. | ¿Toda métrica calculada correctamente resulta útil para decidir? Justifique. |
| Indicador | Es una métrica interpretada en un contexto, con una definición, período, unidad y comparación. | ¿Qué debe añadirse a una métrica para que constituya un indicador? |
| KPI | Es un indicador prioritario que mide el avance hacia un objetivo estratégico. No se deben tener demasiados porque se pierde el enfoque. | ¿Por qué una organización no puede sostener veinte KPI simultáneos? |
| Meta | Es el resultado esperado para un período futuro. El valor observado muestra lo ocurrido; la meta indica lo que se desea alcanzar. | ¿Qué distingue un valor observado en la base de una meta propuesta para el ejercicio? |
| North Star | Es la métrica principal que representa el valor que la organización entrega al cliente y guía sus decisiones. | ¿Qué condición debe cumplir una North Star para no convertirse en métrica de vanidad? |
| Driver | Es una variable que influye en una métrica principal. Se valida mediante análisis de datos, evidencia histórica, experimentos o análisis causal. | ¿Cómo se comprueba que una variable es efectivamente driver de la North Star? |
| Guardrail | Es una métrica de control que evita efectos negativos al mejorar la North Star, como reducir el margen o aumentar las devoluciones. | ¿Qué ocurre si una organización optimiza su North Star sin vigilar los guardrails? |

**Criterio de cierre.** No se avanza al desarrollo práctico mientras existan conceptos sin definición registrada.


## Actividad 2 — Desarrollo práctico y ejecución

### Presentación del caso

Una empresa minorista en línea del Reino Unido cuenta con un registro histórico de transacciones que incluye facturas, productos, cantidades, fechas, precios unitarios, clientes y países. La dirección necesita transformar estos registros en un sistema de métricas que permita distinguir crecimiento útil de crecimiento aparente. Para ello, no basta con observar ventas acumuladas o cantidad de clientes: es necesario identificar qué indicadores representan valor recurrente para el cliente y cuáles pueden orientar una decisión empresarial.

Durante el laboratorio, el equipo trabajará con el conjunto Online Retail del UCI Machine Learning Repository. A partir de los datos reales, deberá auditar y preparar las transacciones, identificar clientes recurrentes, proponer y justificar una North Star, descomponerla en drivers y guardrails, construir un diccionario de KPI y elaborar un tablero de decisión en Plotly. El análisis deberá terminar con hallazgos cuantitativos y una acción empresarial concreta, diferenciando en todo momento los valores observados en la base de las metas o umbrales académicos propuestos para el ejercicio.


## Paso 1 — Preparación reproducible del entorno


In [1]:
%pip install -q ucimlrepo==0.0.7 polars==1.17.1 duckdb==1.1.3

import polars as pl
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from ucimlrepo import fetch_ucirepo

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(160)

FORMATO_FECHA = "%m/%d/%Y %H:%M"

print("polars:", pl.__version__, "| duckdb:", duckdb.__version__)
print("Entorno listo.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 29.8 MB/s eta 0:00:00
polars: 1.17.1 | duckdb: 1.1.3
Entorno listo.


## Paso 2 — Descarga de datos reales desde UCI

El conjunto **Online Retail** contiene transacciones de una empresa minorista en línea registrada en el Reino Unido entre diciembre de 2010 y diciembre de 2011. Los códigos de factura que empiezan con `C` representan cancelaciones. No se generan registros artificiales.


In [2]:
online_retail = fetch_ucirepo(id=352)
df = pl.from_pandas(online_retail.data.original)   # la descarga llega en pandas; se pasa a Polars

print("Dataset:", online_retail.metadata.get("name"))
print("Filas y columnas:", df.shape)
print("Columnas:", df.columns)
display(df.head())


Dataset: Online Retail
Filas y columnas: (541909, 8)
Columnas: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
str,str,str,i64,str,f64,f64,str
"""536365""","""85123A""","""WHITE HANGING HEART T-LIGHT HO…",6,"""12/1/2010 8:26""",2.55,17850.0,"""United Kingdom"""
"""536365""","""71053""","""WHITE METAL LANTERN""",6,"""12/1/2010 8:26""",3.39,17850.0,"""United Kingdom"""
"""536365""","""84406B""","""CREAM CUPID HEARTS COAT HANGER""",8,"""12/1/2010 8:26""",2.75,17850.0,"""United Kingdom"""
"""536365""","""84029G""","""KNITTED UNION FLAG HOT WATER B…",6,"""12/1/2010 8:26""",3.39,17850.0,"""United Kingdom"""
"""536365""","""84029E""","""RED WOOLLY HOTTIE WHITE HEART.""",6,"""12/1/2010 8:26""",3.39,17850.0,"""United Kingdom"""


### Control de trazabilidad
Registre: número de filas, columnas, rango de fechas y países presentes.

**Respuesta:** El dataset descargado tiene 541,909 filas y 8 columnas (`InvoiceNo`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `UnitPrice`, `CustomerID`, `Country`). El rango de fechas va del 1 de diciembre de 2010 (08:26) al 9 de diciembre de 2011 (12:50). Hay 38 países presentes, con el Reino Unido concentrando la gran mayoría de las filas (495,478 de 541,909, ≈91.4%), seguido de Alemania, Francia e Irlanda muy por debajo en volumen.


## Paso 3 — Auditoría inicial y limpieza documentada


In [3]:
df = df.with_columns([
    pl.col("InvoiceNo").cast(pl.Utf8),
    pl.col("CustomerID").cast(pl.Utf8),
    pl.col("Quantity").cast(pl.Float64, strict=False),
    pl.col("UnitPrice").cast(pl.Float64, strict=False),
    pl.col("InvoiceDate").cast(pl.Utf8)
      .str.to_datetime(format=FORMATO_FECHA, strict=False).alias("InvoiceDate"),
])
df = df.with_columns([
    pl.col("InvoiceNo").str.to_uppercase().str.starts_with("C").alias("es_cancelacion"),
    (pl.col("Quantity") * pl.col("UnitPrice")).alias("importe_linea"),
])

# Control de parseo: si el formato declarado no fuese el correcto, aqui apareceria
# un conteo de fechas nulas. pandas habria "adivinado" un formato sin avisar;
# Polars obliga a declararlo y deja el error a la vista.
nulas = df["InvoiceDate"].null_count()
print("Fechas que no pudieron parsearse:", nulas)
if nulas:
    raise ValueError("El formato declarado en FORMATO_FECHA no corresponde a la fuente.")

# Control de integridad de la fuente. Si UCI republica el archivo, el laboratorio
# se detiene aqui en vez de producir numeros que no coinciden con el solucionario.
FILAS_ESPERADAS = 541_909
FACTURAS_ESPERADAS = 25_900

if df.height != FILAS_ESPERADAS:
    raise ValueError(
        f"La fuente cambio: se esperaban {FILAS_ESPERADAS:,} filas y llegaron {df.height:,}. "
        "Avise al docente antes de continuar."
    )
if df["InvoiceNo"].n_unique() != FACTURAS_ESPERADAS:
    raise ValueError(
        f"La fuente cambio: se esperaban {FACTURAS_ESPERADAS:,} facturas distintas "
        f"y llegaron {df['InvoiceNo'].n_unique():,}."
    )
print("Integridad verificada:", f"{df.height:,}", "filas y",
      f"{df['InvoiceNo'].n_unique():,}", "facturas, como se esperaba.")

control = pl.DataFrame({
    "indicador": ["filas", "facturas", "clientes", "cancelaciones",
                  "cantidades_no_positivas", "precios_no_positivos"],
    "valor": [df.height,
              df["InvoiceNo"].n_unique(),
              df["CustomerID"].drop_nulls().n_unique(),
              int(df["es_cancelacion"].sum()),
              int((df["Quantity"] <= 0).sum()),
              int((df["UnitPrice"] <= 0).sum())],
})
display(control)
print("Rango:", df["InvoiceDate"].min(), "->", df["InvoiceDate"].max())


Fechas que no pudieron parsearse: 0
Integridad verificada: 541,909 filas y 25,900 facturas, como se esperaba.


indicador,valor
str,i64
"""filas""",541909
"""facturas""",25900
"""clientes""",4372
"""cancelaciones""",9288
"""cantidades_no_positivas""",10624
"""precios_no_positivos""",2517


Rango: 2010-12-01 08:26:00 -> 2011-12-09 12:50:00


In [4]:
compras = df.filter(
    (~pl.col("es_cancelacion"))
    & (pl.col("Quantity") > 0)
    & (pl.col("UnitPrice") > 0)
    & pl.col("InvoiceDate").is_not_null()
    & pl.col("CustomerID").is_not_null()
)

print("Filas originales:", df.height)
print("Filas de compra validas:", compras.height)
print("Proporcion conservada: {:.1%}".format(compras.height / df.height))


Filas originales: 541909
Filas de compra validas: 397884
Proporcion conservada: 73.4%


### Reto de calidad
1. ¿Qué sesgo aparece si contamos cancelaciones como ventas?  
2. ¿Por qué no debemos borrar esos registros de la fuente original?

**Respuesta:** (1) Las cancelaciones se registran con cantidades negativas (facturas cuyo `InvoiceNo` empieza con "C"). Si se cuentan como ventas normales, el ingreso y el volumen de compras quedan subestimados en la suma bruta (una venta y su cancelación posterior se compensan o incluso invierten el signo del total), y además se mezclan dos eventos de negocio distintos —una compra y un arrepentimiento— bajo el mismo indicador, lo que hace imposible medir la tasa de cancelación como guardrail independiente. (2) No deben borrarse de la fuente original porque son eventos reales que ocurrieron: borrarlos elimina evidencia necesaria para auditar el dato (por ejemplo, para reconciliar por qué el ingreso bruto de un mes no coincide con el ingreso neto reportado a contabilidad). La práctica correcta es excluirlas del subconjunto de análisis (`compras` válidas) sin eliminarlas de la fuente, tal como hace este laboratorio.


## Paso 4 — Definición del periodo comparable

La fuente empieza el 1/12/2010 y termina el 9/12/2011. Para no comparar meses incompletos con meses completos, el laboratorio informa sobre **enero–noviembre de 2011**.

**Cuidado con el sesgo de ventana.** La tabla de facturas se construye sobre *todo* el historial disponible, y el recorte a la ventana se aplica **después** de determinar la recurrencia. Si se recorta antes, un cliente que compró en diciembre de 2010 y volvió en enero de 2011 aparece como comprador nuevo, y la North Star crece de forma artificial en los primeros meses: en esta base, enero pasa de 570 a 246 compras recurrentes, un 57 % menos, solo por haber filtrado en el orden equivocado.


In [5]:
compras = compras.with_columns(pl.col("InvoiceDate").dt.strftime("%Y-%m").alias("mes"))
INICIO, FIN = pl.datetime(2011, 1, 1), pl.datetime(2011, 12, 1)

# La tabla se construye sobre TODO el historial: el recorte a la ventana
# comparable se hace mas adelante, una vez determinada la recurrencia.
facturas = (compras.group_by(["InvoiceNo", "CustomerID", "mes"])
            .agg([pl.col("InvoiceDate").min().alias("fecha_factura"),
                  pl.col("importe_linea").sum().alias("importe_factura"),
                  pl.col("Quantity").sum().alias("unidades"),
                  pl.col("StockCode").count().alias("lineas")]))

en_ventana = facturas.filter((pl.col("fecha_factura") >= INICIO) & (pl.col("fecha_factura") < FIN))
print("Facturas en el historial completo :", facturas["InvoiceNo"].n_unique())
print("Facturas en la ventana comparable :", en_ventana["InvoiceNo"].n_unique())
print("Clientes en la ventana            :", en_ventana["CustomerID"].n_unique())
print("Ingresos en la ventana (GBP)      :", round(en_ventana["importe_factura"].sum(), 2))
display(facturas.head())


Facturas en el historial completo : 18532
Facturas en la ventana comparable : 16354
Clientes en la ventana            : 4173
Ingresos en la ventana (GBP)      : 7820501.22


InvoiceNo,CustomerID,mes,fecha_factura,importe_factura,unidades,lineas
str,str,str,datetime[μs],f64,f64,u32
"""570862""","""17528.0""","""2011-10""",2011-10-12 15:35:00,467.36,360.0,28
"""572231""","""13974.0""","""2011-10""",2011-10-21 14:41:00,413.59,249.0,28
"""543009""","""18041.0""","""2011-02""",2011-02-02 13:11:00,128.68,122.0,10
"""542258""","""17372.0""","""2011-01""",2011-01-26 17:08:00,162.4,58.0,20
"""572886""","""12448.0""","""2011-10""",2011-10-26 13:46:00,449.45,243.0,22


## Paso 5 — Recurrencia y North Star

**Regla operativa del laboratorio:** un cliente se considera recurrente desde su segunda factura válida, contada sobre **todo el historial disponible**. Su primera compra no se reclasifica retrospectivamente.

Enero de 2011 conserva un sesgo residual, porque solo dispone de un mes previo de historial. Por eso se marca como **mes de calentamiento** y se excluye de las comparaciones de variación mensual.


In [6]:
facturas = facturas.sort(["CustomerID", "fecha_factura", "InvoiceNo"])
facturas = facturas.with_columns(
    (pl.int_range(pl.len()).over("CustomerID") + 1).alias("n_compra_cliente"))

# Una factura es recurrente a partir de la SEGUNDA compra del cliente, en orden
# cronologico. La marca se calcula sobre el historial completo para no clasificar
# retroactivamente como recurrente la primera compra de un cliente que volvio despues.
facturas = facturas.with_columns(
    (pl.col("n_compra_cliente") >= 2).alias("es_recurrente"))

# Recien ahora se recorta a la ventana comparable: la recurrencia ya quedo
# determinada usando todo el historial, sin sesgo de ventana.
facturas = facturas.filter((pl.col("fecha_factura") >= INICIO) & (pl.col("fecha_factura") < FIN))

# Enero solo tiene un mes previo de historial: no es comparable en variacion.
MES_CALENTAMIENTO = "2011-01"

mensual = (facturas.group_by("mes")
           .agg([pl.col("InvoiceNo").n_unique().alias("facturas_validas"),
                 pl.col("CustomerID").n_unique().alias("clientes_activos"),
                 pl.col("importe_factura").sum().alias("ingresos")]))

rec = (facturas.filter("es_recurrente").group_by("mes")
       .agg([pl.col("InvoiceNo").n_unique().alias("compras_recurrentes"),
             pl.col("CustomerID").n_unique().alias("clientes_recurrentes"),
             pl.col("importe_factura").sum().alias("ingresos_recurrentes")]))

kpi = mensual.join(rec, on="mes", how="left").fill_null(0).sort("mes")
kpi = kpi.with_columns([
    (pl.col("compras_recurrentes") / pl.col("clientes_recurrentes")).alias("frecuencia_recurrente"),
    (100 * pl.col("ingresos_recurrentes") / pl.col("ingresos")).alias("participacion_ingreso_recurrente_pct"),
])
display(kpi)


mes,facturas_validas,clientes_activos,ingresos,compras_recurrentes,clientes_recurrentes,ingresos_recurrentes,frecuencia_recurrente,participacion_ingreso_recurrente_pct
str,u32,u32,f64,u32,u32,f64,f64,f64
"""2011-01""",987,741,569445.04,570,370,296614.01,1.540541,52.088259
"""2011-02""",997,758,447137.35,617,412,299937.36,1.497573,67.079469
"""2011-03""",1321,974,595500.76,869,563,411030.13,1.543517,69.022604
"""2011-04""",1149,856,469200.361,849,587,357273.97,1.446337,76.145289
"""2011-05""",1555,1056,678594.56,1271,809,566039.71,1.571075,83.413535
"""2011-06""",1393,991,661213.69,1151,773,570140.0,1.489004,86.226285
"""2011-07""",1331,949,600091.011,1143,781,532074.68,1.463508,88.665664
"""2011-08""",1280,935,645343.9,1111,778,568263.68,1.428021,88.055947
"""2011-09""",1755,1266,952838.382,1456,996,806684.471,1.461847,84.661207


### North Star propuesta
**Compras válidas de clientes recurrentes por mes.**

Justifique con los cuatro criterios: valor para el cliente, vínculo con valor empresarial, capacidad de influencia del equipo y descomposición en drivers.

**Respuesta:**
- **Valor para el cliente:** una compra recurrente solo ocurre si el cliente ya recibió valor suficiente en su primera compra como para volver — es una señal directa de satisfacción, no una suposición.
- **Vínculo con valor empresarial:** la participación del ingreso recurrente sobre el ingreso total crece de forma sostenida en el periodo, de 52.1% en enero a 90.4% en noviembre — cada vez más del negocio depende de clientes que ya confían en la marca, lo cual es más barato de sostener que adquirir clientes nuevos todo el tiempo.
- **Capacidad de influencia del equipo:** a diferencia de variables externas (el clima, la competencia), los equipos de marketing, CRM y atención al cliente pueden actuar directamente sobre la recurrencia con campañas de reactivación, programas de fidelidad o mejoras de servicio.
- **Descomposición en drivers:** la North Star se descompone exactamente en clientes recurrentes × frecuencia de compra por cliente, sin residuo (el Paso 6 verifica que `compras_recurrentes = clientes_recurrentes × frecuencia_recurrente` con error de reconstrucción de 0 en prácticamente todos los meses) — es una hipótesis de gestión que sí se puede auditar con los datos disponibles.


### Reto 1 — ¿Vanidad o acción?
Clasifique: productos totales, clientes activos mensuales, compras recurrentes, ingresos acumulados, tasa de cancelación y países con ventas. Para cada una indique qué decisión permite tomar.

**Respuesta:**
- **Productos totales (4,070 distintos):** vanidad. Es un número de catálogo; no indica si esos productos se venden, ni a quién, ni permite decidir nada por sí solo.
- **Clientes activos mensuales:** de vanidad parcial. Mezcla clientes nuevos con recurrentes en un solo número — sube con cualquier tipo de actividad y no distingue si el negocio está creciendo de forma sana o solo captando compradores de una sola vez.
- **Compras recurrentes (North Star):** accionable. Cuando cae, hay una decisión clara: activar retención sobre los clientes que dejaron de comprar.
- **Ingresos acumulados:** vanidad si se reporta como cifra histórica total sin periodo — crece con el tiempo casi por definición y no dice si el mes actual fue bueno o malo.
- **Tasa de cancelación:** accionable (guardrail). Si sube por encima del promedio del periodo (16.76%), la decisión es revisar causas de devolución con el equipo de operaciones.
- **Países con ventas (38):** vanidad. Describe alcance geográfico, no calidad ni intención de la demanda; no cambia ninguna decisión comercial por sí sola.


# ☕ RECESO — 8:30 p. m. a 8:45 p. m.

## Paso 6 — Árbol de métricas

El árbol es una **hipótesis de gestión**, no una demostración causal.

**Objetivo → North Star → drivers → guardrails**

- Objetivo: incrementar valor recurrente sin deteriorar calidad.
- North Star: compras válidas de clientes recurrentes / mes.
- Driver 1: clientes recurrentes activos.
- Driver 2: frecuencia de compra por recurrente.
- Guardrail 1: tasa de cancelación.
- Guardrail 2: concentración de ingresos en Top 10 clientes.


In [7]:
# Validacion aritmetica del primer nivel del arbol
kpi = kpi.with_columns(
    (pl.col("clientes_recurrentes") * pl.col("frecuencia_recurrente")).alias("ns_reconstruida"))
kpi = kpi.with_columns(
    (pl.col("compras_recurrentes") - pl.col("ns_reconstruida")).alias("error_reconstruccion"))
display(kpi.select(["mes", "compras_recurrentes", "clientes_recurrentes",
                    "frecuencia_recurrente", "error_reconstruccion"]))


mes,compras_recurrentes,clientes_recurrentes,frecuencia_recurrente,error_reconstruccion
str,u32,u32,f64,f64
"""2011-01""",570,370,1.540541,0.0
"""2011-02""",617,412,1.497573,0.0
"""2011-03""",869,563,1.543517,0.0
"""2011-04""",849,587,1.446337,1.1369e-13
"""2011-05""",1271,809,1.571075,0.0
"""2011-06""",1151,773,1.489004,0.0
"""2011-07""",1143,781,1.463508,0.0
"""2011-08""",1111,778,1.428021,0.0
"""2011-09""",1456,996,1.461847,0.0


## Paso 7 — Guardrail 1: tasa de cancelación


In [8]:
base_total = (df.filter(pl.col("InvoiceDate").is_not_null()
                        & pl.col("CustomerID").is_not_null()
                        & (pl.col("InvoiceDate") >= INICIO)
                        & (pl.col("InvoiceDate") < FIN))
                .with_columns(pl.col("InvoiceDate").dt.strftime("%Y-%m").alias("mes")))

inv_total = base_total.select(["mes", "InvoiceNo", "CustomerID", "es_cancelacion"]).unique()
cancel = (inv_total.group_by("mes")
          .agg([pl.col("InvoiceNo").n_unique().alias("facturas_total"),
                pl.col("es_cancelacion").sum().alias("facturas_canceladas")])
          .with_columns((100 * pl.col("facturas_canceladas") / pl.col("facturas_total"))
                        .alias("tasa_cancelacion_pct"))
          .sort("mes"))
display(cancel)


mes,facturas_total,facturas_canceladas,tasa_cancelacion_pct
str,u32,u32,f64
"""2011-01""",1236,249,20.145631
"""2011-02""",1202,204,16.971714
"""2011-03""",1619,298,18.406424
"""2011-04""",1384,235,16.979769
"""2011-05""",1849,294,15.900487
"""2011-06""",1707,314,18.394845
"""2011-07""",1593,262,16.446955
"""2011-08""",1544,263,17.033679
"""2011-09""",2078,322,15.495669


## Paso 8 — Guardrail 2: concentración Top 10 clientes


In [9]:
cliente_mes = (facturas.group_by(["mes", "CustomerID"])
               .agg(pl.col("importe_factura").sum().alias("ingreso_cliente")))

ordenado = cliente_mes.sort(["mes", "ingreso_cliente"], descending=[False, True])
ordenado = ordenado.with_columns((pl.int_range(pl.len()).over("mes") + 1).alias("ranking"))

conc = (ordenado.group_by("mes")
        .agg([pl.col("ingreso_cliente").sum().alias("ingreso_mes"),
              pl.col("ingreso_cliente").filter(pl.col("ranking") <= 10)
                .sum().alias("ingreso_top10")])
        .with_columns((100 * pl.col("ingreso_top10") / pl.col("ingreso_mes"))
                      .alias("concentracion_top10_pct"))
        .sort("mes"))

display(conc)


mes,ingreso_mes,ingreso_top10,concentracion_top10_pct
str,f64,f64,f64
"""2011-01""",569445.04,193292.89,33.944082
"""2011-02""",447137.35,89670.06,20.054254
"""2011-03""",595500.76,113585.39,19.073929
"""2011-04""",469200.361,73375.7,15.638458
"""2011-05""",678594.56,128119.53,18.880129
"""2011-06""",661213.69,193184.16,29.2166
"""2011-07""",600091.011,121223.38,20.200833
"""2011-08""",645343.9,160995.32,24.947213
"""2011-09""",952838.382,233643.58,24.520799


In [10]:
kpi = (kpi
       .join(cancel.select(["mes", "tasa_cancelacion_pct"]), on="mes", how="left")
       .join(conc.select(["mes", "concentracion_top10_pct"]), on="mes", how="left")
       .sort("mes"))

# Variacion mensual de la North Star y de sus drivers
kpi = kpi.with_columns([
    (100 * (pl.col("compras_recurrentes") / pl.col("compras_recurrentes").shift(1) - 1)).alias("var_ns_pct"),
    (100 * (pl.col("clientes_recurrentes") / pl.col("clientes_recurrentes").shift(1) - 1)).alias("var_clientes_rec_pct"),
    (100 * (pl.col("frecuencia_recurrente") / pl.col("frecuencia_recurrente").shift(1) - 1)).alias("var_frecuencia_pct"),
])
display(kpi)


mes,facturas_validas,clientes_activos,ingresos,compras_recurrentes,clientes_recurrentes,ingresos_recurrentes,frecuencia_recurrente,participacion_ingreso_recurrente_pct,ns_reconstruida,error_reconstruccion,tasa_cancelacion_pct,concentracion_top10_pct,var_ns_pct,var_clientes_rec_pct,var_frecuencia_pct
str,u32,u32,f64,u32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2011-01""",987,741,569445.04,570,370,296614.01,1.540541,52.088259,570.0,0.0,20.145631,33.944082,null,null,null
"""2011-02""",997,758,447137.35,617,412,299937.36,1.497573,67.079469,617.0,0.0,16.971714,20.054254,8.245614,11.351351,-2.789133
"""2011-03""",1321,974,595500.76,869,563,411030.13,1.543517,69.022604,869.0,0.0,18.406424,19.073929,40.842788,36.650485,3.067901
"""2011-04""",1149,856,469200.361,849,587,357273.97,1.446337,76.145289,849.0,1.1369e-13,16.979769,15.638458,-2.301496,4.262877,-6.295983
"""2011-05""",1555,1056,678594.56,1271,809,566039.71,1.571075,83.413535,1271.0,0.0,15.900487,18.880129,49.705536,37.819421,8.624412
"""2011-06""",1393,991,661213.69,1151,773,570140.0,1.489004,86.226285,1151.0,0.0,18.394845,29.2166,-9.441385,-4.449938,-5.223907
"""2011-07""",1331,949,600091.011,1143,781,532074.68,1.463508,88.665664,1143.0,0.0,16.446955,20.200833,-0.695048,1.034929,-1.712256
"""2011-08""",1280,935,645343.9,1111,778,568263.68,1.428021,88.055947,1111.0,0.0,17.033679,24.947213,-2.79965,-0.384123,-2.424841
"""2011-09""",1755,1266,952838.382,1456,996,806684.471,1.461847,84.661207,1456.0,0.0,15.495669,24.520799,31.053105,28.020566,2.368791


### Pregunta 2
Identifique el mes con mayor tasa de cancelación y el mes con mayor concentración Top 10. ¿Qué riesgo representa cada uno?

**Respuesta:** Entre los meses comparables (excluyendo enero, mes de calentamiento), marzo tiene la mayor tasa de cancelación (18.41%), muy cerca de junio (18.39%). Junio, a su vez, tiene la mayor concentración Top 10 del periodo comparable (29.22%). El riesgo de la cancelación es de calidad de la operación o del producto: una tasa alta erosiona el ingreso ya reconocido y señala fricción en la experiencia de compra. El riesgo de la concentración es de dependencia: casi un tercio del ingreso de junio depende de solo 10 clientes, así que perder a algunos de ellos tendría un impacto desproporcionado sobre el ingreso total de ese mes.


## Paso 9 — Comparación reproducible con Polars y DuckDB


In [11]:
ranking_caida = (kpi
                 .select(["mes", "compras_recurrentes", "var_ns_pct",
                          "tasa_cancelacion_pct", "concentracion_top10_pct"])
                 .sort("var_ns_pct"))
ranking_caida


mes,compras_recurrentes,var_ns_pct,tasa_cancelacion_pct,concentracion_top10_pct
str,u32,f64,f64,f64
"""2011-01""",570,null,20.145631,33.944082
"""2011-06""",1151,-9.441385,18.394845,29.2166
"""2011-08""",1111,-2.79965,17.033679,24.947213
"""2011-04""",849,-2.301496,16.979769,15.638458
"""2011-07""",1143,-0.695048,16.446955,20.200833
"""2011-10""",1571,7.898352,14.759169,21.768432
"""2011-02""",617,8.245614,16.971714,20.054254
"""2011-09""",1456,31.053105,15.495669,24.520799
"""2011-03""",869,40.842788,18.406424,19.073929


In [12]:
# DuckDB consulta el DataFrame de Polars directamente.
consulta = duckdb.sql("""
SELECT mes,
       compras_recurrentes,
       ROUND(var_ns_pct, 1) AS var_ns_pct,
       ROUND(tasa_cancelacion_pct, 1) AS cancelacion_pct,
       ROUND(concentracion_top10_pct, 1) AS concentracion_top10_pct
FROM kpi
ORDER BY var_ns_pct ASC NULLS LAST
""").pl()
display(consulta)


mes,compras_recurrentes,var_ns_pct,cancelacion_pct,concentracion_top10_pct
str,u32,f64,f64,f64
"""2011-06""",1151,-9.4,18.4,29.2
"""2011-08""",1111,-2.8,17.0,24.9
"""2011-04""",849,-2.3,17.0,15.6
"""2011-07""",1143,-0.7,16.4,20.2
"""2011-10""",1571,7.9,14.8,21.8
"""2011-02""",617,8.2,17.0,20.1
"""2011-09""",1456,31.1,15.5,24.5
"""2011-03""",869,40.8,18.4,19.1
"""2011-11""",2334,48.6,13.9,15.7


## Paso 10 — Tablero de decisión


In [13]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=kpi["mes"].to_list(), y=kpi["compras_recurrentes"].to_list(),
                         mode="lines+markers", name="North Star"))

# El tablero incorpora el driver principal junto al resultado: un tablero que
# muestra la North Star sin su driver no permite decidir donde actuar.
fig.add_trace(go.Scatter(x=kpi["mes"].to_list(), y=kpi["clientes_recurrentes"].to_list(),
                         mode="lines+markers", name="Clientes recurrentes (driver)"))

fig.update_layout(title="North Star y driver principal - 2011", xaxis_title="Mes",
                  yaxis_title="Cantidad", hovermode="x unified")
fig.show()


In [14]:
fig2 = px.line(kpi.to_pandas(), x="mes", y=["tasa_cancelacion_pct", "concentracion_top10_pct"],
               markers=True, title="Guardrails observados - cancelacion y concentracion")
fig2.update_layout(yaxis_title="Porcentaje (%)", xaxis_title="Mes", legend_title_text="Guardrail")
fig2.show()


In [15]:
fig3 = px.line(kpi.to_pandas(), x="mes", y=["participacion_ingreso_recurrente_pct"],
               markers=True, title="Participacion del ingreso proveniente de compras recurrentes")
fig3.update_layout(yaxis_title="Porcentaje (%)", xaxis_title="Mes")
fig3.show()


### Interpretación obligatoria
No basta con decir "la línea bajó". Responda:
1. ¿Cuánto cambió y en qué periodo?
2. ¿Cómo se comportó el driver principal?
3. ¿Algún guardrail empeoró al mismo tiempo?
4. ¿Qué puede afirmarse como evidencia y qué es solo una inferencia?
5. ¿Quién debería decidir y qué acción ejecutaría?

**Respuesta:**
1. La North Star cayó -9.44% en junio de 2011 frente a mayo, la mayor caída mensual de todo el periodo comparable.
2. El driver principal (clientes recurrentes) cayó -4.45% en el mismo mes, y la frecuencia de compra por cliente recurrente cayó -5.22% — ambos componentes de la North Star empeoraron a la vez.
3. Sí: la concentración Top 10 subió a 29.22%, la más alta entre los meses comparables, y la tasa de cancelación se mantuvo elevada (18.39%, por encima del promedio del periodo).
4. Evidencia: los tres indicadores (North Star, clientes recurrentes, frecuencia) cayeron simultáneamente en junio, y los dos guardrails empeoraron en el mismo mes — esto es un hecho verificable en la tabla. Inferencia: la coincidencia sugiere que junio tuvo un problema operativo generalizado (servicio, disponibilidad de producto, o una campaña que atrajo compradores puntuales grandes en vez de recurrentes) — pero el dataset no contiene la causa exacta, así que afirmar cuál fue el motivo específico sería una opinión, no un hecho.
5. El responsable de retención/CRM debería revisar qué cambió operativamente en junio 2011 (catálogo, tiempos de entrega, atención al cliente) y decidir si se necesita una campaña de reactivación dirigida a los clientes recurrentes que dejaron de comprar ese mes.


## Reto 2 — Diagnóstico sin confundir correlación con causalidad


In [16]:
# Mes con mayor caida porcentual de la North Star.
# Se excluye el mes de calentamiento y el primero comparado contra el.
peor = (kpi.filter((pl.col("mes") > MES_CALENTAMIENTO) & pl.col("var_ns_pct").is_not_null())
        .sort("var_ns_pct")
        .head(1))
print("Mayor caida mensual de la North Star")
display(peor.select(["mes", "var_ns_pct", "var_clientes_rec_pct", "var_frecuencia_pct",
                     "tasa_cancelacion_pct", "concentracion_top10_pct"]))


Mayor caida mensual de la North Star


mes,var_ns_pct,var_clientes_rec_pct,var_frecuencia_pct,tasa_cancelacion_pct,concentracion_top10_pct
str,f64,f64,f64,f64,f64
"""2011-06""",-9.441385,-4.449938,-5.223907,18.394845,29.2166


A partir de la salida anterior, escriba una conclusión en tres capas:

- **Evidencia:** lo que muestran los datos.
- **Inferencia:** explicación plausible, sin afirmar causalidad.
- **Decisión:** acción concreta que debería evaluar el responsable.

**Respuesta:**
- **Evidencia:** junio de 2011 combina la mayor caída de la North Star del periodo (-9.44%), una caída simultánea en clientes recurrentes (-4.45%) y en frecuencia (-5.22%), junto con el guardrail de concentración Top 10 en su punto más alto entre los meses comparables (29.22%).
- **Inferencia:** la combinación de menos clientes recurrentes, menor frecuencia y mayor dependencia de pocos clientes grandes sugiere que junio no fue una caída aislada de demanda, sino un mes donde el negocio dependió más de compradores grandes puntuales que de la base recurrente habitual — sin que el dato permita afirmar la causa raíz exacta.
- **Decisión:** el responsable comercial debería revisar el detalle de pedidos de junio 2011 para confirmar si la caída de clientes recurrentes coincide con algún evento identificable (rotura de stock, cambio de precio, incidencia de servicio), y evaluar una campaña de reactivación dirigida específicamente a los clientes recurrentes que no volvieron a comprar ese mes.


## Diccionario de KPI — completar

| KPI | Fórmula / unidad | Fuente / frecuencia | Responsable | Meta / alerta | Acción |
|---|---|---|---|---|---|
| Compras recurrentes/mes | Nº de facturas de clientes con ≥2 compras en su historial / mes | UCI / mensual | Gerente Comercial | Crecimiento mensual > 0% | Si cae, activar campaña de reactivación sobre clientes recurrentes inactivos |
| Clientes recurrentes activos | Nº de clientes distintos con ≥2 compras en su historial que compraron ese mes | UCI / mensual | Jefe de CRM / Fidelización | Mantener o crecer mes a mes | Segmentar y contactar a los clientes recurrentes que dejaron de comprar |
| Frecuencia recurrente | Compras recurrentes ÷ clientes recurrentes (compras por cliente/mes) | UCI / mensual | Jefe de CRM | ≥ 1.5 compras por cliente recurrente al mes (referencia histórica del periodo) | Si cae, revisar disponibilidad de catálogo y experiencia de recompra |
| Tasa de cancelación | Facturas canceladas ÷ facturas totales × 100 | UCI / mensual | Atención al Cliente / Operaciones | Alerta si supera el promedio del periodo (16.76%) | Revisar causas de devolución (calidad, error de despacho, demora) |
| Concentración Top 10 | Ingreso de los 10 clientes principales ÷ ingreso total del mes × 100 | UCI / mensual | Gerente Comercial | Alerta si supera 25% (umbral pedagógico) | Diversificar cartera de clientes si se supera de forma sostenida |

> Las metas y alertas que propone son supuestos académicos de gestión, no metas oficiales de la empresa.


---

## Reto de aplicación y retroalimentación

En esta sección se aplicarán los procedimientos desarrollados durante la sesión a nuevas situaciones de análisis. Cada ejercicio requiere modificar, completar o construir código a partir de las tablas ya procesadas. Posteriormente, los resultados obtenidos deberán interpretarse brevemente desde una perspectiva empresarial. El propósito es comprobar la comprensión de las técnicas utilizadas y fortalecer la capacidad de adaptar el análisis ante nuevas preguntas de negocio.

**Indicaciones generales.** Los ejercicios operan sobre `kpi_reto`, copia de trabajo de la tabla de KPI, y sobre `facturas`, ya construida en la Actividad 2. Los datos proceden íntegramente del repositorio UCI; no corresponde generar ni sustituir valores en ningún caso. Cada respuesta escrita no debe exceder cuatro líneas.

**Duración en sesión:** 25 minutos. Los ejercicios que no concluyan se completan como avance del entregable.


In [17]:
# Copia de trabajo para la sección de retos.
kpi_reto = kpi.clone()
print("Copia de trabajo creada:", kpi_reto.shape)
print("Columnas disponibles:", kpi_reto.columns)
display(kpi_reto.head())


Copia de trabajo creada: (11, 16)
Columnas disponibles: ['mes', 'facturas_validas', 'clientes_activos', 'ingresos', 'compras_recurrentes', 'clientes_recurrentes', 'ingresos_recurrentes', 'frecuencia_recurrente', 'participacion_ingreso_recurrente_pct', 'ns_reconstruida', 'error_reconstruccion', 'tasa_cancelacion_pct', 'concentracion_top10_pct', 'var_ns_pct', 'var_clientes_rec_pct', 'var_frecuencia_pct']


mes,facturas_validas,clientes_activos,ingresos,compras_recurrentes,clientes_recurrentes,ingresos_recurrentes,frecuencia_recurrente,participacion_ingreso_recurrente_pct,ns_reconstruida,error_reconstruccion,tasa_cancelacion_pct,concentracion_top10_pct,var_ns_pct,var_clientes_rec_pct,var_frecuencia_pct
str,u32,u32,f64,u32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2011-01""",987,741,569445.04,570,370,296614.01,1.540541,52.088259,570.0,0.0,20.145631,33.944082,null,null,null
"""2011-02""",997,758,447137.35,617,412,299937.36,1.497573,67.079469,617.0,0.0,16.971714,20.054254,8.245614,11.351351,-2.789133
"""2011-03""",1321,974,595500.76,869,563,411030.13,1.543517,69.022604,869.0,0.0,18.406424,19.073929,40.842788,36.650485,3.067901
"""2011-04""",1149,856,469200.361,849,587,357273.97,1.446337,76.145289,849.0,1.1369e-13,16.979769,15.638458,-2.301496,4.262877,-6.295983
"""2011-05""",1555,1056,678594.56,1271,809,566039.71,1.571075,83.413535,1271.0,0.0,15.900487,18.880129,49.705536,37.819421,8.624412


### Ejercicio 1 — Construcción de una tasa de recurrencia mensual

La North Star del laboratorio se expresa en cantidad de compras recurrentes, magnitud absoluta que crece cuando aumenta la actividad total del negocio. Una magnitud absoluta, sin embargo, no permite distinguir si la recurrencia mejora o si simplemente hay más transacciones de cualquier tipo.

La tasa de recurrencia corrige esa limitación: expresa qué proporción de las facturas del mes corresponde a clientes que ya habían comprado con anterioridad. Se trata de una magnitud relativa y, por tanto, comparable entre meses de distinto volumen.

**Se solicita:**

1. Calcular, para cada mes, la cantidad total de facturas y la cantidad de facturas recurrentes a partir de `facturas`.
2. Construir la tasa de recurrencia mensual expresada en porcentaje e incorporarla a `kpi_reto`.
3. Identificar el mes con la tasa más alta y el mes con la tasa más baja.
4. Comparar el ordenamiento por tasa de recurrencia con el ordenamiento por compras recurrentes.

**Tiempo estimado:** 5 minutos.


In [18]:
# Ejercicio 1
kpi_reto = kpi_reto.with_columns(
    (100 * pl.col("compras_recurrentes") / pl.col("facturas_validas")).alias("tasa_recurrencia_pct")
)

display(kpi_reto.select(["mes", "facturas_validas", "compras_recurrentes", "tasa_recurrencia_pct"]))

print("Mes con mayor tasa de recurrencia:")
print(kpi_reto.sort("tasa_recurrencia_pct", descending=True).head(1).select(["mes", "tasa_recurrencia_pct"]))
print("Mes con menor tasa de recurrencia:")
print(kpi_reto.sort("tasa_recurrencia_pct").head(1).select(["mes", "tasa_recurrencia_pct"]))

print("\nRanking por compras_recurrentes (top 3):")
print(kpi_reto.sort("compras_recurrentes", descending=True).head(3).select(["mes", "compras_recurrentes"]))
print("Ranking por tasa_recurrencia_pct (top 3):")
print(kpi_reto.sort("tasa_recurrencia_pct", descending=True).head(3).select(["mes", "tasa_recurrencia_pct"]))


mes,facturas_validas,compras_recurrentes,tasa_recurrencia_pct
str,u32,u32,f64
"""2011-01""",987,570,57.75076
"""2011-02""",997,617,61.885657
"""2011-03""",1321,869,65.783497
"""2011-04""",1149,849,73.890339
"""2011-05""",1555,1271,81.736334
"""2011-06""",1393,1151,82.627423
"""2011-07""",1331,1143,85.875282
"""2011-08""",1280,1111,86.796875
"""2011-09""",1755,1456,82.962963


Mes con mayor tasa de recurrencia:
shape: (1, 2)
┌─────────┬──────────────────────┐
│ mes     ┆ tasa_recurrencia_pct │
│ ---     ┆ ---                  │
│ str     ┆ f64                  │
╞═════════╪══════════════════════╡
│ 2011-11 ┆ 87.843432            │
└─────────┴──────────────────────┘
Mes con menor tasa de recurrencia:
shape: (1, 2)
┌─────────┬──────────────────────┐
│ mes     ┆ tasa_recurrencia_pct │
│ ---     ┆ ---                  │
│ str     ┆ f64                  │
╞═════════╪══════════════════════╡
│ 2011-01 ┆ 57.75076             │
└─────────┴──────────────────────┘

Ranking por compras_recurrentes (top 3):
shape: (3, 2)
┌─────────┬─────────────────────┐
│ mes     ┆ compras_recurrentes │
│ ---     ┆ ---                 │
│ str     ┆ u32                 │
╞═════════╪═════════════════════╡
│ 2011-11 ┆ 2334                │
│ 2011-10 ┆ 1571                │
│ 2011-09 ┆ 1456                │
└─────────┴─────────────────────┘
Ranking por tasa_recurrencia_pct (top 3):
shape: (

### Pregunta 3
¿El mes con más compras recurrentes en términos absolutos es también el de mayor tasa de recurrencia? ¿Qué implica si no coinciden?

**Respuesta:** Coinciden solo en el primer lugar: noviembre lidera ambos rankings (mayor volumen absoluto y mayor tasa, 87.84%). Pero el resto del ranking no coincide — por volumen absoluto, el 2.º y 3.º lugar son octubre y septiembre; por tasa, son agosto y julio. Esto implica que octubre y septiembre tuvieron mucho volumen total (incluyendo bastantes clientes nuevos), mientras que julio y agosto, con menos volumen total, tuvieron una proporción más alta de compras que sí eran recurrentes. Mirar solo el número absoluto de compras recurrentes escondería que la "calidad" de esa recurrencia —qué proporción de las ventas del mes son de clientes que ya confían en la marca— no siempre acompaña al volumen total.


### Ejercicio 2 — Incorporación del ticket promedio como driver económico

El árbol de métricas descompuso la North Star en clientes recurrentes y frecuencia de compra. Ninguno de esos dos drivers recoge el valor económico de cada transacción: dos meses con idéntica cantidad de compras recurrentes pueden presentar ingresos muy distintos si el importe promedio de la factura cambia.

El ticket promedio de la factura recurrente completa esa descripción y permite establecer si el crecimiento observado proviene de mayor actividad o de mayor valor por transacción.

**Se solicita:**

1. Calcular, a partir de `facturas`, el importe promedio de las facturas recurrentes de cada mes.
2. Incorporar el resultado a `kpi_reto` en la columna `ticket_promedio_recurrente`.
3. Calcular su variación mensual porcentual, siguiendo el mismo procedimiento empleado para los demás drivers.
4. Contrastar los meses de mayor caída de la North Star con el comportamiento del ticket promedio en esos mismos meses.

**Tiempo estimado:** 5 minutos.


In [19]:
# Ejercicio 2
kpi_reto = kpi_reto.with_columns(
    (pl.col("ingresos_recurrentes") / pl.col("compras_recurrentes")).alias("ticket_promedio_recurrente")
)
kpi_reto = kpi_reto.with_columns(
    (100 * (pl.col("ticket_promedio_recurrente") / pl.col("ticket_promedio_recurrente").shift(1) - 1))
    .alias("var_ticket_pct")
)

display(kpi_reto.select(["mes", "ticket_promedio_recurrente", "var_ticket_pct", "var_ns_pct"]))

print("Comportamiento del ticket promedio en el mes de mayor caida de la North Star (2011-06):")
print(kpi_reto.filter(pl.col("mes") == "2011-06").select(["mes", "var_ns_pct", "ticket_promedio_recurrente", "var_ticket_pct"]))


mes,ticket_promedio_recurrente,var_ticket_pct,var_ns_pct
str,f64,f64,f64
"""2011-01""",520.375456,null,null
"""2011-02""",486.122139,-6.582424,8.245614
"""2011-03""",472.992094,-2.700977,40.842788
"""2011-04""",420.817397,-11.030776,-2.301496
"""2011-05""",445.34989,5.829724,49.705536
"""2011-06""",495.34318,11.225621,-9.441385
"""2011-07""",465.507157,-6.023304,-0.695048
"""2011-08""",511.488461,9.877679,-2.79965
"""2011-09""",554.041532,8.319459,31.053105


Comportamiento del ticket promedio en el mes de mayor caida de la North Star (2011-06):
shape: (1, 4)
┌─────────┬────────────┬────────────────────────────┬────────────────┐
│ mes     ┆ var_ns_pct ┆ ticket_promedio_recurrente ┆ var_ticket_pct │
│ ---     ┆ ---        ┆ ---                        ┆ ---            │
│ str     ┆ f64        ┆ f64                        ┆ f64            │
╞═════════╪════════════╪════════════════════════════╪════════════════╡
│ 2011-06 ┆ -9.441385  ┆ 495.34318                  ┆ 11.225621      │
└─────────┴────────────┴────────────────────────────┴────────────────┘


### Pregunta 4
En el mes de mayor caída de la North Star, ¿el ticket promedio también cayó? ¿Qué le dice eso sobre la causa del problema?

**Respuesta:** No — en junio de 2011, mientras la North Star caía -9.44%, el ticket promedio recurrente en realidad **subió** +11.23%. Esto es clave: la caída no fue porque los clientes recurrentes gastaran menos por compra. Fue porque hubo **menos clientes recurrentes comprando** (-4.45%) y **con menor frecuencia** (-5.22%) — los que sí compraron, gastaron más que el mes anterior. Eso apunta a un problema de retención o activación de clientes (menos gente volviendo a comprar), no a un problema de valor percibido o de precio por transacción.


### Ejercicio 3 — Definición de un guardrail de dependencia del cliente principal

La Actividad 2 incorporó un guardrail de concentración sobre los diez principales clientes. Ese umbral describe la dependencia agregada del negocio, pero no revela si dicha concentración se explica por un único cliente de gran tamaño, situación que constituye un riesgo operativo de naturaleza distinta.

Un guardrail de dependencia del cliente principal mide qué proporción del ingreso mensual corresponde al cliente de mayor facturación. Su finalidad no es optimizarse, sino advertir cuando la concentración alcanza un nivel que compromete la continuidad del negocio.

**Se solicita:**

1. Calcular, para cada mes, el ingreso del cliente de mayor facturación. La tabla `ordenado` ya dispone de la columna `ranking` calculada dentro de cada mes.
2. Expresar ese ingreso como porcentaje del ingreso total del mes e incorporarlo a `kpi_reto`.
3. Declarar explícitamente un umbral de alerta como criterio pedagógico del ejercicio, no como norma sectorial.
4. Identificar los meses que superan dicho umbral.

**Tiempo estimado:** 5 minutos.


In [20]:
#Ejercicio 3
principal = (ordenado.filter(pl.col("ranking") == 1)
             .select(["mes", "ingreso_cliente"])
             .rename({"ingreso_cliente": "ingreso_cliente_principal"}))

kpi_reto = kpi_reto.join(principal, on="mes", how="left")
kpi_reto = kpi_reto.with_columns(
    (100 * pl.col("ingreso_cliente_principal") / pl.col("ingresos")).alias("dependencia_cliente_principal_pct")
)

UMBRAL_DEPENDENCIA = 15.0

display(kpi_reto.select(["mes", "ingreso_cliente_principal", "ingresos", "dependencia_cliente_principal_pct"]))

sobre_umbral = kpi_reto.filter(pl.col("dependencia_cliente_principal_pct") > UMBRAL_DEPENDENCIA)
print(f"\nMeses que superan el umbral declarado ({UMBRAL_DEPENDENCIA}%):", sobre_umbral.height)
print(sobre_umbral.select(["mes", "dependencia_cliente_principal_pct"]))


mes,ingreso_cliente_principal,ingresos,dependencia_cliente_principal_pct
str,f64,f64,f64
"""2011-01""",77183.6,569445.04,13.554179
"""2011-02""",22797.46,447137.35,5.098536
"""2011-03""",21462.4,595500.76,3.604093
"""2011-04""",21535.9,469200.361,4.589915
"""2011-05""",28408.14,678594.56,4.18632
"""2011-06""",41959.44,661213.69,6.345821
"""2011-07""",26464.99,600091.011,4.410163
"""2011-08""",40327.81,645343.9,6.249042
"""2011-09""",75412.64,952838.382,7.914526



Meses que superan el umbral declarado (15.0%): 0
shape: (0, 2)
┌─────┬─────────────────────────────────┐
│ mes ┆ dependencia_cliente_principal_… │
│ --- ┆ ---                             │
│ str ┆ f64                             │
╞═════╪═════════════════════════════════╡
└─────┴─────────────────────────────────┘


### Pregunta 5
¿Algún mes supera el umbral de dependencia que definió? ¿Qué decisión empresarial sugiere ese resultado?

**Respuesta:** Con un umbral de 15% declarado para este ejercicio, ningún mes lo supera — el más cercano es enero (13.55%, aunque es el mes de calentamiento y su cifra es menos confiable por el sesgo de historial corto). Esto sugiere que, en 2011, ningún mes dependió de forma crítica de un solo cliente según este criterio, así que no hay evidencia que justifique una acción de mitigación urgente sobre este riesgo específico en el periodo analizado. Aun así, vale la pena seguir monitoreando el indicador — varios meses de mitad de año rondan el 6-8%, una dependencia no despreciable que podría crecer si la base recurrente se debilita, como pasó en junio.


### Ejercicio 4 — Consulta de meses saludables con condiciones múltiples en DuckDB

En la Actividad 2 se empleó DuckDB para ordenar los meses según la variación de la North Star. Una consulta orientada a la decisión, sin embargo, no se limita a ordenar: delimita el subconjunto de periodos que satisfacen simultáneamente el objetivo de crecimiento y las restricciones fijadas por los guardrails.

Un mes de crecimiento acompañado de un deterioro en la tasa de cancelación no constituye un mes saludable. Esta consulta materializa esa distinción en una regla reproducible.

**Se solicita:**

1. Construir sobre `kpi_reto` una consulta SQL que incluya `SELECT`, `WHERE`, condiciones múltiples enlazadas con `AND` y `ORDER BY`.
2. Establecer como criterios una variación de la North Star positiva y una tasa de cancelación inferior al promedio del periodo.
3. Ordenar el resultado por variación de la North Star de mayor a menor.
4. Determinar cuántos meses satisfacen ambos criterios.

**Tiempo estimado:** 5 minutos.


In [21]:
# Ejercicio 4
promedio_cancelacion = kpi_reto["tasa_cancelacion_pct"].mean()
print(f"Tasa de cancelacion promedio del periodo: {promedio_cancelacion:.2f}%")

meses_saludables = duckdb.sql(f"""
SELECT mes,
       ROUND(var_ns_pct, 1) AS var_ns_pct,
       ROUND(tasa_cancelacion_pct, 1) AS tasa_cancelacion_pct
FROM kpi_reto
WHERE var_ns_pct > 0
  AND tasa_cancelacion_pct < {promedio_cancelacion}
ORDER BY var_ns_pct DESC
""").pl()

display(meses_saludables)
print(f"\nMeses que califican como saludables: {meses_saludables.height} de {kpi_reto.filter(pl.col('var_ns_pct').is_not_null()).height} meses comparables")


Tasa de cancelacion promedio del periodo: 16.76%


mes,var_ns_pct,tasa_cancelacion_pct
str,f64,f64
"""2011-05""",49.7,15.9
"""2011-11""",48.6,13.9
"""2011-09""",31.1,15.5
"""2011-10""",7.9,14.8



Meses que califican como saludables: 4 de 10 meses comparables


### Pregunta 6
¿Cuántos meses califican como saludables? ¿Qué le dice ese número sobre la calidad del crecimiento del negocio en el periodo?

**Respuesta:** Solo 4 de los 10 meses comparables califican como saludables (mayo, septiembre, octubre y noviembre) — es decir, apenas 4 de cada 10 meses combinaron crecimiento de la North Star con una tasa de cancelación por debajo del promedio del periodo. Eso dice que el crecimiento del negocio en 2011 no fue sostenido ni "limpio": en 6 de los 10 meses, o la North Star no creció, o creció acompañada de una cancelación por encima del promedio — una señal de que buena parte del año, el crecimiento (cuando lo hubo) vino con más fricción operativa de la deseable, no de una base sólida y estable.


### Ejercicio 5 — Representación de la relación entre el driver y la North Star

El tablero construido en la Actividad 2 presenta la North Star y su driver como series temporales paralelas. Esa disposición permite observar la evolución de ambas magnitudes, pero no muestra con claridad si sus variaciones se corresponden entre sí.

Un gráfico de dispersión que enfrente la variación del driver con la variación de la North Star hace visible esa correspondencia: los puntos alineados sobre una tendencia ascendente indican que el driver acompaña el movimiento del resultado, mientras que los puntos dispersos advierten que otros factores intervienen. Corresponde recordar que la correspondencia observada describe una asociación y no acredita una relación causal.

**Se solicita:**

1. Construir con Plotly un gráfico de dispersión que sitúe la variación porcentual de clientes recurrentes en el eje horizontal y la variación porcentual de la North Star en el eje vertical.
2. Identificar cada punto con el mes correspondiente.
3. Rotular los ejes y titular el gráfico de modo que resulte interpretable sin recurrir al código.
4. Excluir de manera explícita el mes de calentamiento, que carece de variación calculable.

**Tiempo estimado:** 5 minutos.


In [22]:
# Ejercicio 5
disp = kpi_reto.filter(pl.col("mes") != MES_CALENTAMIENTO).filter(pl.col("var_ns_pct").is_not_null())

fig5 = go.Figure()
fig5.add_trace(go.Scatter(
    x=disp["var_clientes_rec_pct"].to_list(),
    y=disp["var_ns_pct"].to_list(),
    mode="markers+text",
    text=disp["mes"].to_list(),
    textposition="top center",
    marker=dict(size=11, color="#1F4E96"),
))
fig5.update_layout(
    title="Variacion de clientes recurrentes vs. variacion de la North Star (excluye mes de calentamiento)",
    xaxis_title="Variacion mensual de clientes recurrentes (%)",
    yaxis_title="Variacion mensual de la North Star (%)",
)
fig5.show()

correlacion = disp.select(pl.corr("var_clientes_rec_pct", "var_ns_pct")).item()
print(f"Correlacion entre ambas variaciones: {correlacion:.3f}")


Correlacion entre ambas variaciones: 0.983


### Pregunta 7
¿Qué tan fuerte es la relación entre ambas variables? ¿Hay algún mes que se aparte del patrón general? ¿Cómo lo explicaría?

**Respuesta:** La relación es muy fuerte: la correlación es 0.983, casi perfecta — cuando los clientes recurrentes suben o bajan en un mes, la North Star se mueve en la misma dirección casi siempre, lo cual tiene sentido porque la North Star se construye directamente a partir de ese driver. El mes que más se aparta del patrón es **abril**: los clientes recurrentes subieron ligeramente (+4.26%), pero la North Star cayó (-2.30%) — los signos van en direcciones opuestas, el único caso así en el periodo. La explicación está en el otro driver: en abril la frecuencia recurrente cayó -6.30%, la mayor caída de frecuencia del año, compensando de sobra el leve aumento de clientes. Este caso confirma que la North Star depende de **ambos** drivers (clientes y frecuencia) a la vez, no solo del que tiene mayor correlación visual con ella.


## Informe ejecutivo — síntesis final

Complete cada campo con una frase basada en evidencia de este notebook.

**Objetivo estratégico:** Incrementar el ingreso recurrente del negocio sin comprometer la calidad de la cartera de clientes (concentración) ni el nivel de cancelaciones.

**North Star elegida y por qué:** Compras válidas de clientes recurrentes por mes — porque conecta directamente el valor que recibe el cliente (volver a comprar) con el valor del negocio (el ingreso recurrente pasó de representar 52.1% a 90.4% del ingreso total entre enero y noviembre), y se descompone sin residuo en clientes recurrentes × frecuencia, lo que la hace auditable.

**Principal hallazgo del periodo:** Junio de 2011 concentra la mayor caída de la North Star (-9.44%), con ambos drivers deteriorándose a la vez (clientes recurrentes -4.45%, frecuencia -5.22%) y los dos guardrails empeorando simultáneamente (concentración Top 10 en su punto más alto del periodo comparable, 29.22%; cancelación en 18.39%, por encima del promedio).

**Guardrail más crítico a vigilar:** La concentración Top 10, porque en varios meses (especialmente junio) se acerca a niveles donde perder a un puñado de clientes grandes tendría un impacto desproporcionado sobre el ingreso mensual.

**Calidad del crecimiento observado:** Irregular — solo 4 de 10 meses comparables combinaron crecimiento de la North Star con una cancelación por debajo del promedio del periodo; el negocio creció, pero no de forma consistentemente sana.

**Riesgo no cuantificable con estos datos:** La causa raíz de la caída de junio (¿problema de inventario, de servicio, cambio de precio, estacionalidad?) — el dataset permite ver que varios indicadores empeoraron juntos, pero no explica por qué.

**Decisión recomendada:** Priorizar una investigación operativa de junio 2011 (inventario, tiempos de despacho, incidencias de atención al cliente) antes de lanzar cualquier campaña de adquisición, ya que el problema principal del periodo es de retención de clientes existentes, no de captación.

**Responsable de ejecutar la decisión:** Gerencia Comercial, en conjunto con el equipo de CRM/Fidelización.

**Próxima pregunta que abriría este análisis:** ¿Qué ocurrió operativamente en junio 2011 que explique la caída simultánea de clientes recurrentes y frecuencia? (requeriría cruzar este dataset con registros de inventario, incidencias de servicio o campañas de esa fecha).


## Ticket de salida

Elija una métrica de su proyecto integrador. Explique por qué es accionable y qué decisión cambiaría si disminuyera 20 %.

**Respuesta:**


## Referencias

- Chen, D. (2015). *Online Retail* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33  
- Croll, A., & Yoskovitz, B. (2013). *Lean Analytics*. O’Reilly Media.  
- Parmenter, D. (2020). *Key Performance Indicators* (4th ed.). Wiley.  
- Sharda, R., Delen, D., & Turban, E. (2024). *Business Intelligence, Analytics, Data Science, and AI* (5th ed.). Pearson.
